# Digit Recognizer — Распознавание рукописных цифр (MNIST)

**Тема:** Классификация изображений свёрточной нейросетью (CNN)  
**Инструменты:** `PyTorch`, `torchvision`

---

## О задаче

Классический MNIST: 28×28 картинки рукописных цифр (0–9), нужно предсказать цифру. Данные приходят в виде CSV, где каждая строка — 784 пикселя (28×28), значения 0–255.

**Почему CNN:** свёрточная сеть учится распознавать локальные узоры (края, изгибы) независимо от их положения на картинке — это то, что нужно для изображений, и работает лучше обычного полносвязного слоя.

## 1. Загрузка данных и `Dataset`

Оборачиваем данные в `Dataset`, который:
- нормализует пиксели в диапазон `[0, 1]` (делим на 255);
- приводит каждую строку к форме `(1, 28, 28)` — один канал (ч/б), высота, ширина. Канал обязателен для `Conv2d`.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class MNISTDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.is_test = is_test
        if not is_test:
            self.labels = df['label'].values
            self.features = df.drop('label', axis=1).values.astype('float32') / 255.0
        else:
            self.features = df.values.astype('float32') / 255.0

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        image = torch.tensor(self.features[idx], dtype=torch.float32).view(1, 28, 28)
        if not self.is_test:
            return image, torch.tensor(self.labels[idx], dtype=torch.long)
        return image

Делим обучающий файл на train/val (80/20), чтобы честно проверять качество. Тестовый файл Kaggle без ответов — только для финального предсказания.

In [ ]:
full_train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
train_df, val_df = train_test_split(full_train_df, test_size=0.2, random_state=42)
final_test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

train_loader = DataLoader(MNISTDataset(train_df), batch_size=64, shuffle=True)
val_loader   = DataLoader(MNISTDataset(val_df),   batch_size=64, shuffle=False)
test_loader  = DataLoader(MNISTDataset(final_test_df, is_test=True), batch_size=64, shuffle=False)

print('Train:', len(train_df), '| Val:', len(val_df), '| Test:', len(final_test_df))

## 2. Архитектура сети

Простая CNN: два свёрточных блока (Conv → ReLU → MaxPool), затем линейный слой на 10 классов.

| Слой | Что делает |
|---|---|
| `Conv2d(1→16)` + pool | 16 фильтров, размер падает 28→14 |
| `Conv2d(16→32)` + pool | 32 фильтра, размер падает 14→7 |
| `Linear(32·7·7 → 10)` | классификатор на 10 цифр |

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# 'mps' на Apple Silicon, 'cuda' на Kaggle/Nvidia, иначе 'cpu'
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if torch.backends.mps.is_available() else 'cpu')
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print('Устройство:', device)

## 3. Обучение

На каждой эпохе: прогоняем train (обновляем веса), затем val (только считаем метрики). Следим, чтобы train и val accuracy росли вместе — если train сильно обгоняет val, это переобучение.

In [ ]:
epochs = 5
for epoch in range(epochs):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    train_acc = correct / total

    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            v_correct += (outputs.argmax(1) == labels).sum().item()
            v_total += labels.size(0)
    val_acc = v_correct / v_total

    print(f'Эпоха {epoch+1}/{epochs} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%')

## 4. Финальное предсказание и сабмит

In [ ]:
model.eval()
predictions = []
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        predictions.extend(model(images).argmax(1).cpu().numpy())

submission = pd.DataFrame({'ImageId': range(1, len(predictions) + 1), 'Label': predictions})
submission.to_csv('submission.csv', index=False)
print('submission.csv сохранён!')
submission.head()

## Итог

| Шаг | Что сделали |
|---|---|
| Данные | CSV → нормализация /255 → форма `(1, 28, 28)` |
| Split | train/val 80/20 для честной оценки |
| Модель | 2 свёрточных блока + линейный классификатор |
| Обучение | Adam, CrossEntropyLoss, 5 эпох |
| Метрика | Accuracy на валидации |

**Как улучшить:** больше эпох, аугментации (сдвиги/повороты), Dropout и BatchNorm, глубже сеть.